In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

: 

In [ ]:
data = pd.read_csv("quikr_car.csv")
data.head()

In [ ]:
data.describe()

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
print(data.fuel_type.value_counts())

In [ ]:
data['year'].unique()

In [ ]:
data['Price'].unique()

In [ ]:
data["kms_driven"].unique()

In [ ]:
data['fuel_type'].unique()

## Problems


*   year column has many categorical data
*   Price column has "Ask for Price"
*   kms_driven column needs to omit the string value in each row
*   There are NaN val in fuel_type
*   keep only first 3 words of name column
*   Removing outliers










# Data Pre-Processing

In [ ]:
backup = data.copy()

In [ ]:
data = data[data['year'].str.isnumeric()] #string operation on every row to filter out numeric values

In [ ]:
data['year']=data['year'].astype(int)

In [ ]:
data.info()

In [ ]:
data['Price'] #Price has 'ask for price'

In [ ]:
data = data[data['Price'] != "Ask For Price"]

Since the Price is in object type, we remove the ',' to convert the data into int

In [ ]:
data['Price']=data['Price'].str.replace(',','').astype(int) #replace commas with empty string

In [ ]:
data['kms_driven']=data['kms_driven'].str.split(' ').str.get(0).str.replace(',','')
#split the kms column into two strings and consider only integer values

In [ ]:
data=data[data['kms_driven'].str.isnumeric()]

In [ ]:
data.info()

In [ ]:
#convert kms_driven into int
data['kms_driven'] = data['kms_driven'].astype(int)

In [ ]:
data.info()

In [ ]:
data=data[~data['fuel_type'].isna()]

In [ ]:
data['name'] = data['name'].str.split(' ').str.slice(0,3).str.join(' ')
#split using space and slice first 3 index and join

In [ ]:
data = data.reset_index(drop = True)
#if drop is set to false, all the previous index which is not updated will also be shown

In [ ]:
data.info()

In [ ]:
data.describe() #checking outliers

In [ ]:
data[data['Price']>6e6] #outlier

In [ ]:
data = data[data['Price']<6e6].reset_index(drop= True)

# Cleaned Data

Checking relationship of Company with Price

In [ ]:
plt.subplots(figsize=(15,7))
ax=sns.boxplot(x='company',y='Price',data=data)
ax.set_xticklabels(ax.get_xticklabels(),rotation=40,ha='right')
plt.show()
#the distribution of Price values for each company,
# showing the median, quartiles, and any outliers

Checking relationship of Year with Price

In [ ]:
plt.subplots(figsize=(20,10))
ax=sns.swarmplot(x='year',y='Price',data=data)

Checking relationship of Fuel Type with Price

In [ ]:
plt.subplots(figsize=(14,7))
sns.boxplot(x='fuel_type',y='Price',data=data)

# Model

In [ ]:
X = data.drop(columns='Price') #all columns excluding Price
y = data['Price']

In [ ]:
X

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2)

In [ ]:
ohe = OneHotEncoder()
ohe.fit(X[['name', 'company','fuel_type']])

In [ ]:
#Creating a column transformer to transform categorical columns
column_trans=make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),
                                    remainder='passthrough')
#passthrough is to consider only categorical values and skip other values

In [ ]:
lr = LinearRegression()

In [ ]:
pipe=make_pipeline(column_trans,lr)

In [ ]:
pipe.fit(X_train,y_train)

In [ ]:
y_pred=pipe.predict(X_test)

R2 Score

In [ ]:
r2_score(y_test,y_pred)

Finding the model with a random state of TrainTestSplit

In [ ]:
scores=[]
for i in range(1000):
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=i)
    lr=LinearRegression()
    pipe=make_pipeline(column_trans,lr)
    pipe.fit(X_train,y_train)
    y_pred=pipe.predict(X_test)
    scores.append(r2_score(y_test,y_pred))

The best model is found at random state 302

In [ ]:
np.argmax(scores)

In [ ]:
scores[np.argmax(scores)]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=np.argmax(scores))
lr=LinearRegression()
pipe=make_pipeline(column_trans,lr)
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
r2_score(y_test,y_pred)

In [ ]:
import pickle
pickle.dump(pipe, open('LinearRegressionModel.pkl', 'wb'))
pipe.predict(pd.DataFrame([['Maruti Suzuki Swift', 'Maruti', 2019, 100, 'Petrol']], columns=['name', 'company', 'year', 'kms_driven', 'fuel_type']))